# Redfin Raw Data Quality

**Purpose.** Audit the files ingested for this provider and the corresponding `raw.*`
DuckDB tables before any normalization, blending, or analytical transformation.

This notebook covers the supplied files/tables, observation grain, date and geography
coverage, column types and meanings, missingness and suppression, duplicate/invalid
keys, numeric ranges, suspicious values, source limitations, and downstream readiness.

## Setup and provider rules

In [1]:
from pathlib import Path
import re
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "data" / "quoll.duckdb").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DB_PATH = ROOT / "data" / "quoll.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)

PROVIDER = 'redfin'
TABLE_PATTERNS = ['redfin_%']
PRIMARY_PATTERNS = ['redfin_housing_market_by_county']
KEY_CANDIDATES = [['PERIOD_BEGIN', 'REGION', 'PROPERTY_TYPE']]
DATE_CANDIDATES = ['PERIOD_BEGIN', 'PERIOD_END', 'LAST_UPDATED']
GEO_CANDIDATES = ['REGION', 'STATE_CODE']
NUMERIC_HINTS = ['MEDIAN_SALE_PRICE', 'MEDIAN_PPSF', 'MEDIAN_PPSF_YOY', 'HOMES_SOLD', 'INVENTORY', 'MEDIAN_DOM']
SUPPRESSION_CODES = ['', '-888888888', '-999999999', 'null']

def matches(name, patterns):
    return any(re.fullmatch(pattern.replace("%", ".*"), name, flags=re.I) for pattern in patterns)

def qi(value):
    return '"' + value.replace('"', '""') + '"'

raw_tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'raw' ORDER BY table_name"
).df()["table_name"].tolist()
provider_tables = [name for name in raw_tables if matches(name, TABLE_PATTERNS)]
primary_tables = [name for name in provider_tables if matches(name, PRIMARY_PATTERNS)]
provider_tables, primary_tables

(['redfin_housing_market_by_county'], ['redfin_housing_market_by_county'])

## Files and tables supplied

In [2]:
file_inventory = con.execute(
    '''
    SELECT table_name, filename, source_folder, source_path,
           loaded_at, row_count, detected_columns,
           upstream_source_url, content_sha256
    FROM meta.files
    WHERE table_schema = 'raw'
    ORDER BY table_name
    '''
).df()
file_inventory = file_inventory.loc[file_inventory["table_name"].isin(provider_tables)]

table_rows = []
for table in provider_tables:
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    column_count = con.execute(
        "SELECT count(*) FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).fetchone()[0]
    table_rows.append({"table_name": table, "rows": row_count, "columns": column_count,
                       "primary_data_table": table in primary_tables})
table_inventory = pd.DataFrame(table_rows)
display(file_inventory)
display(table_inventory)

,table_name,filename,source_folder,source_path,loaded_at,row_count,detected_columns,upstream_source_url,content_sha256
55,redfin_housing_market_by_county,Redfin-Housing-Market-By-County.csv,housing,C:\Users\kenny\Data Science Projects\quoll-int...,2026-07-28T16:55:35.168520+00:00,1307833,"[""column00"", ""PERIOD_BEGIN"", ""PERIOD_END"", ""PE...",https://www.redfin.com/news/data-center/,0569e48c8d45c4d9115283f7d33f17e564a165b2ce203c...


,table_name,rows,columns,primary_data_table
0,redfin_housing_market_by_county,1307833,59,True


## Observation grain

One county-period-property-type housing-market observation.

The checks below infer candidate keys from the raw columns. A repeated candidate key is
reported rather than silently removed because some provider tables legitimately contain
additional dimensions.

## Column types and meanings

In [3]:
schema_frames = []
for table in primary_tables:
    schema = con.execute(f"DESCRIBE raw.{qi(table)}").df()
    schema.insert(0, "table_name", table)
    schema["inferred_meaning"] = (
        schema["column_name"].str.replace("_", " ", regex=False)
        .str.replace(r"(?<=[a-z])(?=[A-Z])", " ", regex=True)
        .str.strip()
    )
    schema_frames.append(schema)
schema_inventory = pd.concat(schema_frames, ignore_index=True) if schema_frames else pd.DataFrame()
display(schema_inventory)

,table_name,column_name,column_type,null,key,default,extra,inferred_meaning
0,redfin_housing_market_by_county,column00,VARCHAR,YES,None,None,None,column00
1,redfin_housing_market_by_county,PERIOD_BEGIN,VARCHAR,YES,None,None,None,PERIOD BEGIN
2,redfin_housing_market_by_county,PERIOD_END,VARCHAR,YES,None,None,None,PERIOD END
3,redfin_housing_market_by_county,PERIOD_DURATION,VARCHAR,YES,None,None,None,PERIOD DURATION
4,redfin_housing_market_by_county,REGION_TYPE,VARCHAR,YES,None,None,None,REGION TYPE
5,redfin_housing_market_by_county,REGION_TYPE_ID,VARCHAR,YES,None,None,None,REGION TYPE ID
6,redfin_housing_market_by_county,TABLE_ID,VARCHAR,YES,None,None,None,TABLE ID
7,redfin_housing_market_by_county,IS_SEASONALLY_ADJUSTED,VARCHAR,YES,None,None,None,IS SEASONALLY ADJUSTED
8,redfin_housing_market_by_county,REGION,VARCHAR,YES,None,None,None,REGION
9,redfin_housing_market_by_county,CITY,VARCHAR,YES,None,None,None,CITY


## Date and geographic coverage

In [4]:
coverage_rows = []
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"].tolist()
    row = {"table_name": table}
    for column in DATE_CANDIDATES:
        if column in columns:
            normalized_column = column.lower()
            if normalized_column == "year" or normalized_column.endswith("_year"):
                coverage_type = "INTEGER"
            elif normalized_column == "month" or normalized_column.endswith("_month"):
                coverage_type = "INTEGER"
            else:
                coverage_type = "TIMESTAMP"
            values = con.execute(
                f"SELECT min(try_cast({qi(column)} AS {coverage_type})), "
                f"max(try_cast({qi(column)} AS {coverage_type})) "
                f"FROM raw.{qi(table)}"
            ).fetchone()
            row[f"{column}_min"] = values[0]
            row[f"{column}_max"] = values[1]
    for column in GEO_CANDIDATES:
        if column in columns:
            row[f"{column}_distinct"] = con.execute(
                f"SELECT count(DISTINCT {qi(column)}) FROM raw.{qi(table)}"
            ).fetchone()[0]
    coverage_rows.append(row)
coverage = pd.DataFrame(coverage_rows)
display(coverage)

,table_name,PERIOD_BEGIN_min,PERIOD_BEGIN_max,PERIOD_END_min,PERIOD_END_max,LAST_UPDATED_min,LAST_UPDATED_max,REGION_distinct,STATE_CODE_distinct
0,redfin_housing_market_by_county,2012-01-01,2025-12-01,2012-01-31,2025-12-31,None,None,3087,51


## Missingness and suppression codes

In [5]:
missing_rows = []
suppression_rows = []
suppression_sql = ", ".join("?" for _ in SUPPRESSION_CODES)
for table in primary_tables:
    columns = con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=? ORDER BY ordinal_position", [table]
    ).df()["column_name"].tolist()
    row_count = con.execute(f"SELECT count(*) FROM raw.{qi(table)}").fetchone()[0]
    # Profile all columns for compact tables and the first 80 for unusually wide sources.
    for column in columns[:80]:
        null_count, blank_count = con.execute(
            f"SELECT count(*) FILTER (WHERE {qi(column)} IS NULL), "
            f"count(*) FILTER (WHERE trim(cast({qi(column)} AS VARCHAR))='') "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        missing_rows.append({
            "table_name": table, "column_name": column,
            "missing_count": null_count + blank_count,
            "missing_pct": (null_count + blank_count) / row_count * 100 if row_count else np.nan,
        })
        if SUPPRESSION_CODES:
            suppressed = con.execute(
                f"SELECT count(*) FROM raw.{qi(table)} "
                f"WHERE trim(cast({qi(column)} AS VARCHAR)) IN ({suppression_sql})",
                SUPPRESSION_CODES,
            ).fetchone()[0]
            if suppressed:
                suppression_rows.append({
                    "table_name": table, "column_name": column,
                    "suppression_or_sentinel_count": suppressed,
                })
missingness = pd.DataFrame(missing_rows).sort_values(
    ["missing_pct", "table_name"], ascending=[False, True]
)
suppression = pd.DataFrame(suppression_rows)
display(missingness)
display(suppression if not suppression.empty else pd.DataFrame(
    {"result": ["No configured literal suppression codes were present in profiled columns; nulls remain material."]}
))

,table_name,column_name,missing_count,missing_pct
9,redfin_housing_market_by_county,CITY,1307833,100.000000
52,redfin_housing_market_by_county,PRICE_DROPS_YOY,769669,58.850710
51,redfin_housing_market_by_county,PRICE_DROPS_MOM,699334,53.472729
50,redfin_housing_market_by_county,PRICE_DROPS,641224,49.029501
57,redfin_housing_market_by_county,PARENT_METRO_REGION_METRO_CODE,369412,28.246114
46,redfin_housing_market_by_county,AVG_SALE_TO_LIST_YOY,244573,18.700629
34,redfin_housing_market_by_county,NEW_LISTINGS_YOY,234861,17.958027
25,redfin_housing_market_by_county,MEDIAN_LIST_PPSF_YOY,231281,17.684291
19,redfin_housing_market_by_county,MEDIAN_LIST_PRICE_YOY,229792,17.570439
45,redfin_housing_market_by_county,AVG_SALE_TO_LIST_MOM,217180,16.606096


,result
0,No configured literal suppression codes were p...


## Duplicate or invalid keys

In [6]:
key_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    keys = next((candidate for candidate in KEY_CANDIDATES if set(candidate).issubset(columns)), [])
    if not keys:
        key_rows.append({"table_name": table, "candidate_key": None,
                         "duplicate_key_groups": np.nan, "invalid_key_rows": np.nan})
        continue
    key_expr = ", ".join(qi(column) for column in keys)
    invalid = " OR ".join(
        f"{qi(column)} IS NULL OR trim(cast({qi(column)} AS VARCHAR))=''" for column in keys
    )
    duplicate_groups = con.execute(
        f"SELECT count(*) FROM (SELECT {key_expr}, count(*) n "
        f"FROM raw.{qi(table)} GROUP BY {key_expr} HAVING count(*) > 1)"
    ).fetchone()[0]
    invalid_rows = con.execute(
        f"SELECT count(*) FROM raw.{qi(table)} WHERE {invalid}"
    ).fetchone()[0]
    key_rows.append({"table_name": table, "candidate_key": " + ".join(keys),
                     "duplicate_key_groups": duplicate_groups,
                     "invalid_key_rows": invalid_rows})
key_quality = pd.DataFrame(key_rows)
display(key_quality)

,table_name,candidate_key,duplicate_key_groups,invalid_key_rows
0,redfin_housing_market_by_county,PERIOD_BEGIN + REGION + PROPERTY_TYPE,0,0


## Numeric ranges and suspicious values

In [7]:
numeric_rows = []
for table in primary_tables:
    columns = set(con.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema='raw' AND table_name=?", [table]
    ).df()["column_name"])
    for column in [name for name in NUMERIC_HINTS if name in columns]:
        numeric = (
            f"try_cast(replace(trim(cast({qi(column)} AS VARCHAR)), ',', '') AS DOUBLE)"
        )
        result = con.execute(
            f"SELECT count(*) FILTER (WHERE {numeric} IS NOT NULL), "
            f"min({numeric}), max({numeric}), "
            f"count(*) FILTER (WHERE {numeric} < 0) "
            f"FROM raw.{qi(table)}"
        ).fetchone()
        numeric_rows.append({
            "table_name": table, "column_name": column,
            "numeric_count": result[0], "minimum": result[1],
            "maximum": result[2], "negative_count": result[3],
            "review_flag": (
                "review negative values/sentinels" if result[3] else
                "review extreme min/max against provider definition"
            ),
        })
numeric_ranges = pd.DataFrame(numeric_rows)
display(numeric_ranges)

,table_name,column_name,numeric_count,minimum,maximum,negative_count,review_flag
0,redfin_housing_market_by_county,MEDIAN_SALE_PRICE,1306716,1.000000,1.000000e+09,0,review extreme min/max against provider defini...
1,redfin_housing_market_by_county,MEDIAN_PPSF,1294875,0.000277,1.000000e+09,0,review extreme min/max against provider defini...
2,redfin_housing_market_by_county,MEDIAN_PPSF_YOY,1138059,-1.000000,1.584088e+07,366184,review negative values/sentinels
3,redfin_housing_market_by_county,HOMES_SOLD,1306743,1.000000,9.237000e+03,0,review extreme min/max against provider defini...
4,redfin_housing_market_by_county,INVENTORY,1261023,1.000000,2.891900e+04,0,review extreme min/max against provider defini...
5,redfin_housing_market_by_county,MEDIAN_DOM,1296673,1.000000,3.664300e+04,0,review extreme min/max against provider defini...


## Source-specific limitations

Redfin coverage reflects transactions and listings visible to Redfin, county histories are incomplete in some places, revision timing is provider-controlled, and YOY measures require prior-year observations.

## Downstream readiness

**Assessment: PASS WITH LIMITATIONS for All Residential county-month analysis after FIPS resolution, numeric parsing, sentinel removal, and complete-window filtering.**

This assessment is conditional on the displayed inventories and checks. The normalized
`mart.*` builders—not this notebook—own parsing, suppression handling, geographic
resolution, deduplication, and downstream transformations.

In [8]:
con.close()